# HOMER: scalar block means and inference

This notebook examines inference after smooth aggregation of independent block means. A fixed pseudo-Huber threshold defines a block target that need not equal the population mean. The simulations separate target mismatch from the accuracy of the sandwich approximation and from the effect of having few blocks.

The saved experiment has 26 cells and 2,000 replications per cell. Its Bernoulli targets use the entire binomial support; Gaussian cells provide a symmetric reference and fixed-block sensitivity checks.


## Choose the results to read

`paper` reads the compact manuscript summaries without rerunning experiments. Set `MODE` to `smoke` for a short numerical run or `full` for manuscript replication counts. Fresh outputs go to `runs/`; the saved figures below are displayed only in `paper` mode.


In [ ]:
MODE = "paper"  # "paper", "smoke", or "full"
STUDIES = ['scalar']


In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "results").is_dir() and (p / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
if MODE not in {"paper", "smoke", "full"}:
    raise ValueError("MODE must be 'paper', 'smoke', or 'full'.")

if MODE == "paper":
    RESULTS = ROOT / "results"
    display(Markdown("**Reading the saved manuscript results.** No experiments are rerun."))
else:
    subprocess.run(
        [sys.executable, str(ROOT / "code" / "reproduce.py"), "--mode", MODE,
         "--study", *STUDIES, "--resume"],
        cwd=ROOT, check=True,
    )
    RESULTS = ROOT / "runs" / MODE / "results"
    display(Markdown(f"**Reading {MODE} results** from `runs/{MODE}/results/`."))

pd.set_option("display.max_columns", 12)
pd.set_option("display.precision", 4)


## Compare population-mean and block-target coverage

The two rows per setting compare the same estimated-variance interval against two targets. Monte Carlo intervals quantify uncertainty in a simulated coverage proportion; they are not confidence intervals for either target.


In [ ]:
scalar = pd.read_csv(RESULTS / "scalar_summary.csv")
coverage = scalar.loc[
    scalar["metric"].isin(["plugin_mean_covered", "plugin_target_covered"]),
    ["distribution", "schedule", "k", "m", "lambda", "metric", "reps",
     "coverage", "mc_low", "mc_high"],
]
display(coverage.loc[coverage["schedule"].isin(["slow", "boundary", "many"])]
        .reset_index(drop=True))


## Inspect the numerical targets and fit diagnostics

The finite-sample block targets are evaluated before the Monte Carlo comparisons. A small score residual checks the target calculation. Inference records also retain solver, covariance, and certificate failures rather than dropping them.


In [ ]:
targets = pd.read_csv(RESULTS / "scalar_targets.csv")
display(targets[["m", "p", "lambda", "theta", "V", "score_residual"]]
        .drop_duplicates().reset_index(drop=True))

# Failure counts repeat across coverage metrics; count each experimental cell once.
checks = scalar.drop_duplicates("cell")
display(checks[["solver_failures", "covariance_failures", "certificate_failures"]]
        .sum().rename("total").to_frame())


## Manuscript figure

Saved scalar results: scalar coverage in the top row and functional efficiency and contamination comparisons below. The preview is rendered from the figure PDF, with the same fonts and sizes.

In [ ]:
if MODE == "paper":
    display(Image(filename=str(ROOT / "figures" / "exp_overview.png")))
else:
    display(Markdown(
        "Fresh-run summaries are shown above. To regenerate all manuscript-style "
        "figures from a complete run, use `python code/reproduce.py --mode " + MODE + " --figures`. "
        "Figure generation requires pdfLaTeX, PGF, and Poppler."
    ))


## Scope

These results concern fixed deterministic thresholds and block means. They do not provide an adaptive-threshold inference theorem. The settings with few blocks and those with a substantial target difference are part of the comparison, including their undercoverage.

For independent high-precision target checks, run `python code/reproduce.py --mode full --study scalar --audit-targets`. See `../docs/DATA_SOURCES.md` and the saved manifests for the exact seeds, formulas, and computational scope.
